# 📊 LSTM Data Exploration — Sales Dataset
### Stage 1: Exploratory Data Analysis (7 Visualizations)
---
**Visualizations covered:**
1. Time Series Plot of Sales Over Time
2. Rolling Mean & Standard Deviation (Moving Average)
3. Sales Distribution (Histogram + KDE + Box Plot)
4. Seasonal Decomposition (Trend / Seasonality / Residual)
5. Autocorrelation (ACF) & Partial Autocorrelation (PACF)
6. Sales by Category / Segment / Market (Bar Charts)
7. Correlation Heatmap of Numeric Features


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})
BLUE   = "#2563EB"
ORANGE = "#F97316"
RED    = "#EF4444"
GREEN  = "#10B981"
PURPLE = "#8B5CF6"
PALETTE = [BLUE, ORANGE, RED, GREEN, PURPLE, "#EC4899", "#14B8A6"]

print("✅ Libraries loaded successfully")


## 📂 Load & Prepare Data

In [ ]:
# ── UPDATE THIS PATH to your CSV/Excel file ───────────────────────────────────
FILE_PATH = "Global_Superstore2.csv"   # e.g. "superstore.csv"

# ── Load ─────────────────────────────────────────────────────────────────────
try:
    if FILE_PATH.endswith((".xlsx", ".xls")):
        df = pd.read_excel(FILE_PATH)
    else:
        df = pd.read_csv(FILE_PATH, encoding="latin1")
    print(f"✅ Loaded {len(df):,} rows × {len(df.columns)} columns")
except FileNotFoundError:
    # ── Demo mode: generate synthetic data so every cell still runs ──────────
    print("⚠  File not found — generating synthetic demo data")
    np.random.seed(42)
    dates = pd.date_range("2011-01-01", "2014-12-31", freq="D")
    df = pd.DataFrame({
        "Order Date": np.random.choice(dates, 9994),
        "Ship Date":  np.random.choice(dates, 9994),
        "Sales":      np.random.exponential(scale=250, size=9994).clip(1),
        "Quantity":   np.random.randint(1, 15, 9994),
        "Discount":   np.random.choice([0, 0.1, 0.2, 0.3, 0.4, 0.5], 9994),
        "Category":   np.random.choice(["Furniture","Office Supplies","Technology"], 9994,
                                        p=[0.21, 0.60, 0.19]),
        "Segment":    np.random.choice(["Consumer","Corporate","Home Office"], 9994,
                                        p=[0.52, 0.30, 0.18]),
        "Market":     np.random.choice(["US","APAC","EU","LATAM","Africa","EMEA"], 9994,
                                        p=[0.30, 0.20, 0.20, 0.10, 0.10, 0.10]),
        "Sub-Category": np.random.choice(
            ["Chairs","Tables","Binders","Paper","Phones","Machines",
             "Storage","Copiers","Accessories","Supplies"], 9994),
        "Region": np.random.choice(["East","West","Central","South",
                                     "Oceania","North Asia","Southeast Asia"], 9994),
    })

# ── Parse dates & sort ────────────────────────────────────────────────────────
for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

df.sort_values("Order Date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Monthly time-series ───────────────────────────────────────────────────────
ts = df.set_index("Order Date")["Sales"].resample("M").sum()

print(f"Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Monthly TS : {len(ts)} data points")
print("\nFirst 5 rows:")
df.head()


## 📈 Visualization 1 — Time Series of Sales Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))

ax.plot(ts.index, ts.values, color=BLUE, linewidth=1.6, alpha=0.8, label="Monthly Sales")

# Annotate min / max
ax.annotate(f"Peak\n${ts.max():,.0f}",
            xy=(ts.idxmax(), ts.max()),
            xytext=(ts.idxmax(), ts.max() * 1.08),
            ha="center", fontsize=9, color=RED,
            arrowprops=dict(arrowstyle="->", color=RED, lw=1.2))

ax.set_title("Monthly Sales Over Time", pad=12)
ax.set_xlabel("Date"); ax.set_ylabel("Total Sales ($)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("viz1_sales_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Monthly Sales — Min: ${ts.min():,.0f}  |  Max: ${ts.max():,.0f}  |  Mean: ${ts.mean():,.0f}")


## 📉 Visualization 2 — Rolling Mean & Standard Deviation (Moving Average)

In [ ]:
roll3  = ts.rolling(window=3)
roll6  = ts.rolling(window=6)
roll12 = ts.rolling(window=12)

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# ── Top: raw + rolling means ─────────────────────────────────────────────────
axes[0].plot(ts.index, ts.values, color=BLUE, alpha=0.45, linewidth=1.2, label="Raw Monthly Sales")
axes[0].plot(ts.index, roll3.mean(),  color=ORANGE, linewidth=2,   label="3-Month MA")
axes[0].plot(ts.index, roll6.mean(),  color=RED,    linewidth=2,   label="6-Month MA")
axes[0].plot(ts.index, roll12.mean(), color=PURPLE, linewidth=2.2, label="12-Month MA")
axes[0].fill_between(ts.index,
                     roll6.mean() - roll6.std(),
                     roll6.mean() + roll6.std(),
                     alpha=0.12, color=RED, label="6-Mo ±1σ Band")
axes[0].set_title("Rolling Mean (Moving Average)")
axes[0].set_ylabel("Sales ($)")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[0].legend(fontsize=9, ncol=3)

# ── Bottom: rolling std (volatility) ─────────────────────────────────────────
axes[1].plot(ts.index, roll3.std(),  color=ORANGE, linewidth=1.8, label="3-Month Std")
axes[1].plot(ts.index, roll6.std(),  color=RED,    linewidth=1.8, label="6-Month Std")
axes[1].plot(ts.index, roll12.std(), color=PURPLE, linewidth=2,   label="12-Month Std")
axes[1].set_title("Rolling Standard Deviation (Volatility / Noise)")
axes[1].set_ylabel("Std Dev ($)")
axes[1].set_xlabel("Date")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].legend(fontsize=9)

plt.suptitle("Rolling Mean & Standard Deviation Analysis", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("viz2_rolling_mean_std.png", dpi=150, bbox_inches="tight")
plt.show()


## 📊 Visualization 3 — Sales Distribution (Histogram + KDE + Box Plot)

In [ ]:
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, width_ratios=[2, 2, 1], wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# Histogram
ax1.hist(df["Sales"], bins=80, color=BLUE, edgecolor="white", alpha=0.85)
ax1.set_title("Histogram of Individual Sales")
ax1.set_xlabel("Sales ($)"); ax1.set_ylabel("Count")
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# KDE
df["Sales"].plot.kde(ax=ax2, color=BLUE, linewidth=2.5)
ax2.axvline(df["Sales"].mean(),   color=RED,    linestyle="--", linewidth=1.8, label=f'Mean  ${df["Sales"].mean():,.0f}')
ax2.axvline(df["Sales"].median(), color=ORANGE, linestyle="--", linewidth=1.8, label=f'Median ${df["Sales"].median():,.0f}')
ax2.set_title("KDE — Probability Density")
ax2.set_xlabel("Sales ($)"); ax2.set_ylabel("Density")
ax2.legend(fontsize=9)

# Box plot
bp = ax3.boxplot(df["Sales"], patch_artist=True, vert=True,
                 medianprops=dict(color=RED, linewidth=2),
                 boxprops=dict(facecolor=BLUE, alpha=0.5))
ax3.set_title("Box Plot")
ax3.set_ylabel("Sales ($)")
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax3.set_xticks([])

plt.suptitle("Sales Distribution Analysis", fontsize=15, fontweight="bold")
plt.savefig("viz3_sales_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

skew = df["Sales"].skew()
kurt = df["Sales"].kurtosis()
print(f"Skewness: {skew:.3f}  (>1 = right-skewed, typical for sales)")
print(f"Kurtosis: {kurt:.3f}  (>3 = heavy tails / outliers)")
print(f"% of orders > $1,000 : {(df['Sales'] > 1000).mean()*100:.1f}%")


## 🔁 Visualization 4 — Seasonal Decomposition (Trend / Seasonality / Residual)

In [ ]:
ts_clean = ts.dropna()

# Additive decomposition (use multiplicative if variance grows with level)
result_add  = seasonal_decompose(ts_clean, model="additive",      period=12)
result_mult = seasonal_decompose(ts_clean, model="multiplicative", period=12)

fig, axes = plt.subplots(4, 2, figsize=(16, 12))
fig.suptitle("Seasonal Decomposition — Additive (left)  vs  Multiplicative (right)",
             fontsize=14, fontweight="bold", y=1.01)

colors = [BLUE, ORANGE, GREEN, RED]
labels = ["Observed", "Trend", "Seasonal", "Residual"]

for col, result in enumerate([result_add, result_mult]):
    components = [result.observed, result.trend, result.seasonal, result.resid]
    for row, (comp, label, color) in enumerate(zip(components, labels, colors)):
        axes[row][col].plot(comp.index, comp.values, color=color, linewidth=1.6)
        axes[row][col].set_ylabel(label, fontsize=10)
        axes[row][col].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
        if row == 0:
            axes[row][col].set_title("Additive Model" if col == 0 else "Multiplicative Model",
                                     fontsize=12, fontweight="bold")
        if row == 3:
            axes[row][col].axhline(0, color="grey", linestyle="--", linewidth=1)
            axes[row][col].set_xlabel("Date")

plt.tight_layout()
plt.savefig("viz4_seasonal_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()
print("💡 Tip: If seasonal amplitude grows over time → use Multiplicative. Otherwise → Additive.")


## 🔄 Visualization 5 — ACF & PACF (Choose LSTM Look-Back Window)

In [ ]:
ts_clean = ts.dropna()
LAGS = min(24, len(ts_clean) // 2 - 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle("Autocorrelation (ACF) & Partial Autocorrelation (PACF)", fontsize=14,
             fontweight="bold")

# Raw series
plot_acf(ts_clean,  lags=LAGS, ax=axes[0][0], color=BLUE,   alpha=0.05)
axes[0][0].set_title("ACF — Raw Monthly Sales")

plot_pacf(ts_clean, lags=LAGS, ax=axes[0][1], color=ORANGE, alpha=0.05, method="ywm")
axes[0][1].set_title("PACF — Raw Monthly Sales")

# First-differenced (for stationarity)
ts_diff = ts_clean.diff().dropna()
plot_acf(ts_diff,  lags=LAGS, ax=axes[1][0], color=GREEN, alpha=0.05)
axes[1][0].set_title("ACF — First-Differenced Series")

plot_pacf(ts_diff, lags=LAGS, ax=axes[1][1], color=RED,   alpha=0.05, method="ywm")
axes[1][1].set_title("PACF — First-Differenced Series")

for ax in axes.flat:
    ax.set_xlabel("Lag (months)")
    ax.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("viz5_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()

print("💡 Look-back guidance:")
print("   • ACF cuts off at lag k  → MA(k) structure, use k as look-back")
print("   • PACF cuts off at lag k → AR(k) structure, use k as look-back")
print(f"   • For LSTM: try look_back = 12 (one year of monthly data)")


## 🏷️ Visualization 6 — Sales by Category / Segment / Market

In [ ]:
cat_cols = [c for c in ["Category", "Segment", "Market", "Sub-Category", "Region"]
            if c in df.columns]

n = len(cat_cols)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    grouped = df.groupby(col)["Sales"].sum().sort_values(ascending=False)
    bars = ax.bar(range(len(grouped)), grouped.values,
                  color=PALETTE[:len(grouped)], edgecolor="white", linewidth=0.7)

    # Value labels on top
    for bar, val in zip(bars, grouped.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f"${val/1e3:.0f}K", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels(grouped.index, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"Sales by {col}")
    ax.set_ylabel("Total Sales ($)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))

plt.suptitle("Sales Breakdown by Category, Segment & Market", fontsize=15,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz6_sales_by_category.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Bonus: Monthly heatmap (Year × Month) — great for spotting seasonality
df["_Year"]  = df["Order Date"].dt.year
df["_Month"] = df["Order Date"].dt.month

pivot = df.pivot_table(values="Sales", index="_Year", columns="_Month", aggfunc="sum")
pivot.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"][:len(pivot.columns)]

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5,
            linecolor="white", ax=ax,
            annot_kws={"size": 9})
ax.set_title("Monthly Sales Heatmap — Year × Month", fontsize=14, fontweight="bold")
ax.set_xlabel("Month"); ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig("viz6b_monthly_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

df.drop(columns=["_Year", "_Month"], inplace=True, errors="ignore")


## 🔥 Visualization 7 — Correlation Heatmap of Numeric Features

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Add time-based features if Order Date is present
if "Order Date" in df.columns:
    numeric_df["Month"]      = df["Order Date"].dt.month
    numeric_df["Quarter"]    = df["Order Date"].dt.quarter
    numeric_df["DayOfWeek"]  = df["Order Date"].dt.dayofweek
    if "Ship Date" in df.columns:
        numeric_df["ShipLag"] = (df["Ship Date"] - df["Order Date"]).dt.days.clip(0, 60)

corr = numeric_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full heatmap
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, linecolor="white", ax=axes[0],
            annot_kws={"size": 9}, square=True,
            cbar_kws={"shrink": 0.8})
axes[0].set_title("Full Correlation Matrix (Lower Triangle)")
axes[0].tick_params(axis="x", rotation=45)

# Sales-only bar chart
sales_corr = corr["Sales"].drop("Sales").sort_values()
colors_bar  = [RED if v < 0 else GREEN for v in sales_corr.values]
axes[1].barh(sales_corr.index, sales_corr.values, color=colors_bar, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
for i, (idx, val) in enumerate(sales_corr.items()):
    axes[1].text(val + (0.005 if val >= 0 else -0.005), i,
                 f"{val:.3f}", va="center",
                 ha="left" if val >= 0 else "right", fontsize=9)
axes[1].set_title("Correlation with Sales")
axes[1].set_xlabel("Pearson r")
axes[1].set_xlim(-1, 1)

plt.suptitle("Feature Correlation Analysis", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("viz7_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📌 Key correlations with Sales:")
print(corr["Sales"].drop("Sales").sort_values(ascending=False).to_string())


---
## ✅ EDA Summary & Next Steps

| # | Visualization | Key Insight for LSTM |
|---|---------------|----------------------|
| 1 | Sales Over Time | Check for trend direction and jumps |
| 2 | Rolling Mean & Std | Identify window sizes; detect non-stationarity |
| 3 | Distribution | Log-transform if right-skewed before feeding LSTM |
| 4 | Seasonal Decomposition | Determines if seasonality needs to be modeled explicitly |
| 5 | ACF / PACF | Choose `look_back` window (typical: 12 for monthly data) |
| 6 | Category Breakdown | Decide whether to train per-category or a global model |
| 7 | Correlation Heatmap | Select features to add alongside Sales in multi-variate LSTM |

**Recommended LSTM look-back: 12 months**  
**Suggested preprocessing: MinMaxScaler, possibly log1p transform**

---
*Next: Stage 2 — Preprocessing & Train/Val/Test Split*
